# Grocery Sales Forecasting — Walkthrough

End-to-end run of the pipeline against the Kaggle Favorita data, using
the modules in `src/`. Drop the six competition CSVs into `../data/`
first.

Each section here is the same logic that `python -m src.pipeline`
executes, broken up so you can inspect intermediates.


In [ ]:
# Make `src` importable when running from the notebooks/ folder.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config
config.set_seeds()
print('project root:', ROOT)
print('data dir    :', config.DATA_DIR)


## 1. Load + merge the six tables

`build_panel()` concatenates train + test (test rows have NaN sales
and `is_test=1`), joins stores on `store_nbr`, forward-fills oil to a
daily index, joins transactions on (store, date), and normalises the
holiday calendar — `transferred=True` rows are dropped (the holiday
moved away from that date), `Work Day` rows are dropped (make-up
workdays after bridges), and the remainder is split into national /
regional / local flags so each store only inherits the holidays for
its city / state / country.


In [ ]:
from src.data_loader import build_panel

panel = build_panel()
print('panel shape:', panel.shape)
print('date range :', panel['date'].min().date(), '->', panel['date'].max().date())
print('is_test split:', panel['is_test'].value_counts().to_dict())
panel.head()


## 2. Feature engineering (leakage-safe)

Every feature derived from `sales` is shifted by ≥ 16 days. Rolling
windows are computed on the *already-shifted* series so the window
at time t uses sales at t-16, t-17, ..., t-16-w+1.

Calendar / holiday / oil / promo features are known in advance and
used un-lagged.


In [ ]:
from src.features import build_features, feature_columns, CATEGORICAL_COLS

df = build_features(panel)
feat_cols = feature_columns(df)
print(f'{len(feat_cols)} features')
print('sample features:', feat_cols[:20])


## 3. Time-based validation

Two utilities. `final_holdout_split` carves the last 16 days for a
Kaggle-mirroring holdout. `rolling_window_folds` produces N
consecutive 16-day folds stepping backward, with an expanding train
window. Both assert `max(train_date) < min(valid_date)`.


In [ ]:
from src.validation import final_holdout_split, rolling_window_folds

holdout = final_holdout_split(df)
print(f'holdout: train ends {holdout.train_end.date()}'
      f'  valid {holdout.valid_start.date()} .. {holdout.valid_end.date()}')

folds = rolling_window_folds(df, n_folds=4)
for f in folds:
    print(f'  {f.name}: train_end={f.train_end.date()}  valid {f.valid_start.date()}..{f.valid_end.date()}')


## 4. Seasonal-naive baseline (floor we must beat)

Copy the same day-of-week from the last fully-observed week. If our
fancy model can't clear this, the features aren't helping.


In [ ]:
from src.models import SeasonalNaive
from src import evaluate

train_df = df.iloc[holdout.train_idx]
valid_df = df.iloc[holdout.valid_idx]

naive = SeasonalNaive().fit(train_df)
yp_naive = naive.predict(valid_df)
baseline_rmsle = evaluate.rmsle(valid_df['sales'].to_numpy(), yp_naive)
print(f'seasonal-naive RMSLE on holdout: {baseline_rmsle:.5f}')


## 5. LightGBM CV across rolling folds

One global model. Categoricals declared natively. log1p target + RMSE
loss == RMSLE on raw scale.


In [ ]:
from src.models import LGBMConfig, train_lightgbm, predict_lightgbm
import numpy as np, pandas as pd

cfg = LGBMConfig()
cv_records = []
for f in folds:
    X_tr = df.iloc[f.train_idx][feat_cols]
    y_tr = df.iloc[f.train_idx]['sales'].to_numpy()
    X_va = df.iloc[f.valid_idx][feat_cols]
    y_va = df.iloc[f.valid_idx]['sales'].to_numpy()
    res = train_lightgbm(X_tr, y_tr, X_va, y_va, CATEGORICAL_COLS, cfg)
    cv_records.append({'fold': f.name, 'rmsle': res.val_rmsle, 'best_iter': res.best_iteration})
    print(f'  {f.name}: RMSLE={res.val_rmsle:.5f}  best_iter={res.best_iteration}')

cv_df = pd.DataFrame(cv_records)
print(f'\nCV mean RMSLE = {cv_df.rmsle.mean():.5f} +/- {cv_df.rmsle.std():.5f}')


## 6. Final-holdout fit + error analysis

Train on everything before the last 16 days, evaluate on the holdout,
and slice the error by family, store, holiday, and promo.


In [ ]:
X_tr = df.iloc[holdout.train_idx][feat_cols]
y_tr = df.iloc[holdout.train_idx]['sales'].to_numpy()
X_ho = df.iloc[holdout.valid_idx][feat_cols]
y_ho = df.iloc[holdout.valid_idx]['sales'].to_numpy()

final = train_lightgbm(X_tr, y_tr, X_ho, y_ho, CATEGORICAL_COLS, cfg)
yp_ho = predict_lightgbm(final, X_ho)

print(f'holdout RMSLE:           {final.val_rmsle:.5f}')
print(f'holdout MAE:             {evaluate.mae(y_ho, yp_ho):.3f}')
print(f'revenue-weighted MAE:    {evaluate.revenue_weighted_error(y_ho, yp_ho):.3f}')
print(f'baseline seasonal-naive: {baseline_rmsle:.5f}  ({(baseline_rmsle - final.val_rmsle):.5f} better)')


In [ ]:
err = evaluate.error_table(valid_df, y_ho, yp_ho)
print('Holiday vs normal:'); print(evaluate.holiday_vs_normal(err)); print()
print('Promo vs normal:');   print(evaluate.promo_vs_normal(err));   print()
print('Top-10 worst families:'); print(evaluate.worst_segments(err, ['family'])); print()
print('Top-10 worst stores:');   print(evaluate.worst_segments(err, ['store_nbr']))


In [ ]:
fi_path = evaluate.plot_feature_importance(final.feature_importance)
from IPython.display import Image
Image(filename=str(fi_path))


## 7. Final refit on all train data + Kaggle submission

We refit using the `best_iteration` from the holdout (no
early-stopping signal once the validation set is empty) and write the
16-day test predictions.


In [ ]:
from src.pipeline import predict_kaggle
submission = predict_kaggle(df, feat_cols, CATEGORICAL_COLS,
                            final_n_estimators=final.best_iteration,
                            lgbm_cfg=cfg)
submission.head()


Submission is at `outputs/submission.csv`. Upload directly to Kaggle
or run `kaggle competitions submit -c store-sales-time-series-forecasting -f outputs/submission.csv -m "global lightgbm"`.
